# FirstAI — integrated build through Step 3
Run the cells from top to bottom. This notebook verifies the code, checks the full training pipeline on the attached GPU, then provides the persistent real-training workflow. It does not define identity, personality or conversational behaviour.

In [ ]:
import os
if os.path.isdir('/content/FirstAI'):
    !git -C /content/FirstAI pull --ff-only
else:
    !git clone https://github.com/llakshyaa-rgb/FirstAI.git /content/FirstAI
%cd /content/FirstAI
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), 'GiB')

## Test 1 — automated unit tests
Checks tokenizer, corpus preparation, data batching, model shape, causal attention and gradients.

In [ ]:
!python -m unittest discover -s tests -v

## Test 2 — training engine and checkpoint recovery
Runs a tiny temporary training job, saves latest/best checkpoints, resumes, restores scaler/RNG state, evaluates and generates. This is an engineering test, not meaningful knowledge training.

In [ ]:
!python scripts/smoke_test.py --require-cuda

## Test 3 — integrated GPU pipeline with persistent storage
Mount Google Drive so real training data and checkpoints survive a Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/FirstAI')
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'checkpoints'
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print('Persistent data folder:', RAW)
print('Persistent checkpoints folder:', OUT)

### Add your training corpus
Upload UTF-8 `.txt` files you have permission to use. They are stored in your private Google Drive folder, not committed to the public GitHub repository.

In [ ]:
from google.colab import files
uploaded = files.upload()
txt_files = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.txt')]
if not txt_files:
    raise RuntimeError('No .txt files were selected. Upload at least one UTF-8 text file.')
for name, data in txt_files:
    (RAW / Path(name).name).write_bytes(data)
print('Saved', len(txt_files), 'text file(s) to Drive.')

In [ ]:
!python prepare_corpus.py --input-dir "$RAW" --output "$BASE/data/input.txt"
CORPUS = str(BASE / 'data' / 'input.txt')
print('Corpus ready:', CORPUS)

### Train from random initialization
Starter configuration for a T4. This is a small baseline, not a claim that the model will become a capable assistant. Training can be resumed using the next cell.

In [ ]:
CHECKPOINT = str(OUT / 'firstgpt.pt')
!python train.py --data "$CORPUS" --out "$CHECKPOINT" --steps 1000 --eval-interval 100 --checkpoint-interval 250 --batch-size 16 --block-size 128 --layers 4 --heads 4 --width 128 --grad-accum-steps 2 --amp

### Resume training if needed
Run this cell only after a previous training run has created the checkpoint. It uses the same corpus and restores the saved model, optimizer, mixed-precision scaler and RNG states.

In [ ]:
!python train.py --data "$CORPUS" --out "$CHECKPOINT" --resume --steps 1000 --eval-interval 100 --checkpoint-interval 250 --batch-size 16 --grad-accum-steps 2 --amp

### Evaluate and generate
Evaluation checks the exact corpus fingerprint. Generation uses the best validation checkpoint.

In [ ]:
!python evaluate.py --checkpoint "$OUT/firstgpt.best.pt" --data "$CORPUS" --batches 50 --batch-size 16
!python generate.py --checkpoint "$OUT/firstgpt.best.pt" --prompt "The " --tokens 200 --temperature 0.8 --top-k 40

### Download a copy of the checkpoints
The originals remain in Drive. Download these copies if you also want them on your phone.

In [ ]:
from google.colab import files
files.download(CHECKPOINT)
files.download(str(OUT / 'firstgpt.best.pt'))